In [ ]:
# install google library for gemini and the openai library for chatgpt and deepseek
# !pip install -q openai google-generativeai python-dotenv


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# import os
# import pandas as pd
# import json
# import re
# from openai import OpenAI
# from dotenv import load_dotenv
# from pathlib import Path

# # 1. Setup
# load_dotenv()
# client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.getenv("OPEN_ROUTER_KEY"), # Ensure this matches your .env file
# )

# # 2. Configuration
# # Define the base name for your output files
# output_filename_base = "fewshot_5"

# # --- Experiment Setup ---
# models_to_test = {
#     # "ChatGPT": "openai/gpt-5",      # August 7, 2025 version in openrouter
#     # "Gemini": "google/gemini-2.5-pro",        # June 17, 2025 version in openrouter
#     "DeepSeek": "deepseek/deepseek-chat-v3.1"            # August 21, 2025 version in openrouter
# }

# # get the directry of the current script
# base_dir = Path(os.getcwd())

# # go up one level (to the parent directory, then into sister folder 'FewShot')
# input_filename = base_dir.parent / "FewShot" / "inputs" / "fewshot_5_input.json"


# try:
#     with open(input_filename, 'r', encoding='utf-8') as f:
#         # We read it as a string to paste into the prompt
#         json_content_str = f.read() 
# except FileNotFoundError:
#     print(f"❌ Error: Could not find {input_filename}")
#     exit()

# # 4. Construct the Prompt
# # We inject the file content directly into the prompt string.
# # APIs don't have an "upload" button like the web chat; we paste the data into the context.
# base_prompt = """
# Below is a JSON dataset containing 100 natural language stories. 
# The first 5 stories have a valid 'logic_form'. The rest are marked 'N/A'.

# Your Task:
# 1. Analyze the first 5 stories as examples.
# 2. Generate the ASP logic form for the remaining 95 stories.
# 3. Return the COMPLETE JSON with all 100 stories, where 'N/A' is replaced by your generated logic.
# 4. Keep the exact same JSON structure.

# DATASET:
# """

# full_prompt = base_prompt + json_content_str

# # 5. Run Experiment
# print(f"🚀 Sending 100 stories to {len(models_to_test)} models. This may take time...\n")

# for model_name, model_id in models_to_test.items():
#     print(f"⏳ Querying {model_name}...")

#     response_text = None # reset variable so we don't save previous model's output on error
    
#     try:
#         completion = client.chat.completions.create(
#             model=model_id,
#             response_format={"type": "json_object"}, # Force valid JSON
#             messages=[
#                 {"role": "system", "content": "You are an ASP Logic expert. Output strictly valid JSON."},
#                 {"role": "user", "content": full_prompt}
#             ],
#             # crucial: allowing maximum possible output tokens
#             # (If the model supports it, this prevents cutoff)
#             max_tokens=128000 
#         )
        
#         response_text = completion.choices[0].message.content
        
#         # 6. Save the Output
#         # output_filename = base_dir.parent / "FewShot" / f"{output_filename_base}_output_{model_name}.json"
#         # Setup Results Path
#         output_filename = (base_dir.parent / "FewShot" / "Results" / "FewShot_5_V1" / f"{output_filename_base}_output_{model_name}.json").resolve()
#         output_filename = output_filename.parent
#         output_filename.mkdir(parents=True, exist_ok=True)
        
#         # Verify it parses before saving (optional but good for safety)
#         parsed = json.loads(response_text)
        
#         with open(output_filename / f"{output_filename_base}_output_{model_name}.json", 'w', encoding='utf-8') as f:
#             json.dump(parsed, f, indent=4)
            
#         print(f"   ✅ Success! Saved to {output_filename}")

#     except Exception as e:
#         print(f"   ❌ Error with {model_name}: {e}")
#         # If it fails (e.g. token limit), save the raw text anyway so you don't lose the work
#         with open(f"error_raw_{model_name}.txt", "w", encoding='utf-8') as f:
#             if 'response_text' in locals():
#                 f.write(response_text)
#             else:
#                 f.write(str(e))

🚀 Sending 100 stories to 1 models. This may take time...

⏳ Querying DeepSeek...
   ❌ Error with DeepSeek: Unterminated string starting at: line 119 column 22 (char 13040)


In [5]:
import os
import json
import yaml
import math
import random
import time
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from json_repair import repair_json # pip install json-repair
from datetime import datetime, timezone

# --- 1. GLOBAL SETUP & PROMPT BLOCKS ---
load_dotenv()
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_KEY"),
)

SYSTEM_PERSONA = "You are a precise ASP logic translator. You never skip data."

JSON_FORMATTING_RULES = """
CRITICAL RULES:
1. Return ONLY a valid JSON object.
2. The 'logic_form' field MUST be a list of strings.
3. ESCAPE quotes: \"person(\\\"John\\\")\".
4. MANDATORY COMPLETENESS: You MUST process EVERY story provided in the 'TARGET TASK' section. 
5. If I give you 10 stories, you must return exactly 10 logic forms.
"""

# --- 2. HELPER FUNCTIONS ---

def load_file_content(base_dir, relative_path):
    path = (base_dir / relative_path).resolve()
    return path.read_text(encoding='utf-8') if path.exists() else None


def _append_summary_file(summary_path, run_entry):
    # Read existing summary (if any), append new run_entry keyed by run_id
    data = {}
    if summary_path.exists():
        try:
            with open(summary_path, 'r', encoding='utf-8') as f:
                data = yaml.safe_load(f) or {}
        except Exception:
            data = {}
    data[run_entry['run_id']] = run_entry
    with open(summary_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, sort_keys=False)


def run_single_experiment(exp_config, default_models, base_dir):
    exp_id = exp_config.get("experiment_id", "Unnamed")
    print(f"\n🔬 --- Starting Experiment: {exp_id} ---")

    exp_start_time = datetime.now(timezone.utc)

    models_to_run = exp_config.get("models", default_models)
    raw_input = load_file_content(base_dir, exp_config['input_data'])
    if not raw_input:
        print(f"      ❌ Input file not found: {exp_config['input_data']}")
        return
    all_data = json.loads(raw_input)['data']

    # Separate the Few-Shot Examples from the N/A Targets
    examples_list = []
    targets_list = []
    for item in all_data:
        if "logic_form" in item and item['logic_form'] != ["N/A"]:
            examples_list.append({"sid": item['sid'], "text": item['text'], "logic_form": item['logic_form']})
        else:
            targets_list.append({"sid": item['sid'], "text": item['text']})

    print(f"      📊 Found {len(examples_list)} examples and {len(targets_list)} targets")

    # Respect `num_examples` in config (sample if more examples available)
    num_examples = exp_config.get('num_examples', len(examples_list))
    if num_examples and len(examples_list) > num_examples:
        examples_list = random.sample(examples_list, min(num_examples, len(examples_list)))
        print(f"      🎲 Sampled {len(examples_list)} examples (num_examples={num_examples})")

    # Convert examples to string once (used in every batch)
    examples_str = json.dumps(examples_list, indent=2)

    # Setup Results Path
    results_dir = (base_dir.parent / "FewShot" / "Results" / exp_id).resolve()
    results_dir.mkdir(parents=True, exist_ok=True)
    
    # Summary file path (single file for all runs)
    summary_path = (base_dir.parent / "FewShot" / "Results" / "experiments_summary.yaml").resolve()

    # Load Documentation
    docs_content = ""
    for doc_path in exp_config.get('documentation_files', []):
        text = load_file_content(base_dir, doc_path)
        if text:
            docs_content += f"\n\n--- DOCS: {Path(doc_path).name} ---\n{text}"
    
    if exp_config.get('documentation_files'):
        print(f"      📄 Loaded {len(exp_config.get('documentation_files', []))} documentation file(s)")

    # Experiment-level metadata
    exp_meta = {
        'experiment_id': exp_id,
        'input_file': str(exp_config.get('input_data')),
        'num_examples_provided': len(examples_list),
        'num_targets': len(targets_list),
        'start_time_utc': exp_start_time.isoformat(),
        'models': []
    }

    for model_name, model_id in models_to_run.items():
        model_start = datetime.now(timezone.utc)
        print(f"      ⏳ Querying {model_name}...")
        full_results = []

        BATCH_SIZE = exp_config.get('batch_size', 10)
        total_batches = math.ceil(len(targets_list) / BATCH_SIZE)

        model_meta = {
            'model_name': model_name,
            'model_id': model_id,
            'start_time_utc': model_start.isoformat(),
            'batches': [],
            'errors': 0,
            'missing_sids': [],
            'result_file': None
        }

        # Keep track of which SIDs we still need
        target_sids = [t['sid'] for t in targets_list]

        for i in range(0, len(targets_list), BATCH_SIZE):
            batch_num = (i // BATCH_SIZE) + 1
            batch_data = targets_list[i : i + BATCH_SIZE]

            print(f"         > Processing Batch {batch_num}/{total_batches}...")

            # Create a clean list of just the stories for this batch
            task_stories = [{"sid": s["sid"], "text": s["text"]} for s in batch_data]
            batch_input_str = json.dumps(task_stories, indent=2)

            full_prompt = f"""
{JSON_FORMATTING_RULES}

### REFERENCE EXAMPLES:
{examples_str}

### DOCUMENTATION:
{docs_content}

### TARGET TASK:
Translate the following {len(task_stories)} stories into ASP logic.
You must provide a result for EVERY sid listed below.

STORIES TO PROCESS:
{batch_input_str}

### OUTPUT INSTRUCTIONS:
Return a JSON object with the key "data", containing the list of your {len(task_stories)} translations.
Each item must include "sid" and "logic_form".
"""

            response_text = ""
            max_retries = exp_config.get('max_retries', 2)
            success = False
            batch_meta = {'batch_num': batch_num, 'attempts': 0, 'success': False, 'duration_seconds': None, 'error_file': None, 'raw_file': None, 'full_file': None}
            batch_start = datetime.now(timezone.utc)

            for attempt in range(max_retries):
                batch_meta['attempts'] += 1
                try:
                    call_start = time.time()
                    completion = client.chat.completions.create(
                        model=model_id,
                        response_format={"type": "json_object"},
                        messages=[
                            {"role": "system", "content": SYSTEM_PERSONA},
                            {"role": "user", "content": full_prompt}
                        ],
                        max_tokens=32000,
                        temperature=0.0
                    )
                    call_end = time.time()

                    # Save raw response and full completion for debugging
                    try:
                        response_text = completion.choices[0].message.content
                    except Exception:
                        response_text = ''

                    raw_path = results_dir / f"RAW_{model_name}_B{batch_num}.txt"
                    with open(raw_path, 'w', encoding='utf-8') as f:
                        f.write(response_text or '')
                    batch_meta['raw_file'] = str(raw_path)

                    full_path = results_dir / f"FULL_{model_name}_B{batch_num}.json"
                    try:
                        # Try to serialise the completion object
                        comp_obj = completion.to_dict() if hasattr(completion, 'to_dict') else completion.__dict__
                    except Exception:
                        comp_obj = str(completion)
                    with open(full_path, 'w', encoding='utf-8') as f:
                        try:
                            json.dump(comp_obj, f, default=str, indent=2)
                        except Exception:
                            f.write(str(comp_obj))
                    batch_meta['full_file'] = str(full_path)

                    # Repair and Extract
                    repaired_str = repair_json(response_text)
                    parsed = json.loads(repaired_str)

                    # Strict extraction: expect either {'data': [...]} or a list
                    batch_items = None
                    if isinstance(parsed, dict) and "data" in parsed:
                        if isinstance(parsed["data"], list):
                            batch_items = parsed["data"]
                        else:
                            raise ValueError("Expected 'data' to be a list")
                    elif isinstance(parsed, list):
                        batch_items = parsed
                    else:
                        # try to find the first list-like value that looks like items
                        for val in parsed.values() if isinstance(parsed, dict) else []:
                            if isinstance(val, list):
                                batch_items = val
                                break

                    if batch_items is None:
                        raise ValueError("Could not extract list of items from model response")

                    # Basic validation for expected fields
                    valid_items = []
                    for it in batch_items:
                        if isinstance(it, dict) and 'sid' in it and 'logic_form' in it:
                            valid_items.append(it)

                    if not valid_items:
                        raise ValueError("No valid items (with 'sid' and 'logic_form') found in response")

                    full_results.extend(valid_items)
                    success = True
                    batch_meta['success'] = True
                    batch_meta['duration_seconds'] = time.time() - call_start
                    break

                except Exception as e:
                    print(f"         ❗ Attempt {attempt+1}/{max_retries} failed for Batch {batch_num}: {e}")
                    model_meta['errors'] += 1
                    if attempt < max_retries - 1:
                        time.sleep(1)
                        print(f"         🔄 Retrying Batch {batch_num}...")
                        continue
                    else:
                        # Write the raw response / error for debugging
                        err_path = results_dir / f"ERR_{model_name}_B{batch_num}.txt"
                        with open(err_path, 'w', encoding='utf-8') as f:
                            f.write(response_text if response_text else str(e))
                        batch_meta['error_file'] = str(err_path)
                        print(f"         ❌ Batch {batch_num} failed after {max_retries} attempts. Logged to {err_path}")

            batch_meta['duration_seconds'] = batch_meta['duration_seconds'] or ((datetime.now(timezone.utc) - batch_start).total_seconds())
            model_meta['batches'].append(batch_meta)

            if not success:
                # continue to next batch; missing SIDs will be reported later
                continue

        # Post-processing: ensure we have an entry for each target SID and preserve order
        results_map = {}
        for r in full_results:
            sid_val = r.get('sid')
            try:
                sid_key = int(sid_val)
            except Exception:
                sid_key = sid_val
            results_map[sid_key] = r

        final_results = []
        missing_sids = []
        for t in targets_list:
            sid = t['sid']
            if sid in results_map:
                final_results.append(results_map[sid])
            else:
                missing_sids.append(sid)

        model_meta['missing_sids'] = missing_sids

        if missing_sids:
            warn_path = results_dir / f"MISSING_{model_name}.txt"
            with open(warn_path, 'w', encoding='utf-8') as f:
                f.write(f"Missing SIDs after processing: {missing_sids}\n")
            print(f"      ⚠️ Missing {len(missing_sids)} SIDs for {model_name}. See {warn_path.name}")

        # Save Final File (include any examples if desired; we save only generated target results)
        out_file = results_dir / f"{model_name}.json"
        with open(out_file, 'w', encoding='utf-8') as f:
            json.dump({"data": final_results}, f, indent=4)
        model_meta['result_file'] = str(out_file)
        model_meta['end_time_utc'] = datetime.now(timezone.utc).isoformat()
        model_meta['duration_seconds'] = (datetime.now(timezone.utc) - model_start).total_seconds()
        print(f"      ✅ Completed {model_name}: {out_file}")

        exp_meta['models'].append(model_meta)

        # Append a run-level entry per-model to the summary file
        parent_folder = results_dir.parent.parent.name if results_dir.parent.parent else 'UNKNOWN'
        run_id = f"{parent_folder}_{model_name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}"
        run_entry = {
            'run_id': run_id,
            'experiment_id': exp_id,
            'parent_folder': parent_folder,
            'model_name': model_name,
            'model_id': model_id,
            'start_time_utc': model_meta['start_time_utc'],
            'end_time_utc': model_meta['end_time_utc'],
            'duration_seconds': model_meta['duration_seconds'],
            'num_targets': len(targets_list),
            'num_batches': len(model_meta['batches']),
            'errors': model_meta['errors'],
            'missing_sids_count': len(model_meta['missing_sids']),
            'result_file': model_meta['result_file']
        }
        _append_summary_file(summary_path, run_entry)

    exp_meta['end_time_utc'] = datetime.now(timezone.utc).isoformat()
    exp_meta['duration_seconds'] = (datetime.now(timezone.utc) - exp_start_time).total_seconds()

    # Optionally append experiment-level entry as well
    exp_run_id = f"{results_dir.parent.parent.name}_{exp_id}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}"
    exp_entry = {
        'run_id': exp_run_id,
        'experiment_id': exp_id,
        'start_time_utc': exp_meta['start_time_utc'],
        'end_time_utc': exp_meta['end_time_utc'],
        'duration_seconds': exp_meta['duration_seconds'],
        'models': [{'model_name': m['model_name'], 'model_id': m['model_id'], 'duration_seconds': m.get('duration_seconds'), 'errors': m.get('errors'), 'missing_sids_count': len(m.get('missing_sids', []))} for m in exp_meta['models']]
    }
    _append_summary_file(summary_path, exp_entry)

# --- 3. MAIN EXECUTION ---
print("=" * 70)
print("🚀 STARTING EXPERIMENT RUNNER")
print("=" * 70)

current_dir = Path(os.getcwd())
print(f"📍 Current working directory: {current_dir}")

config_path = current_dir / "master_config.yaml"
print(f"🔍 Looking for config at: {config_path}")

if not config_path.exists():
    print(f"❌ Config file NOT found: {config_path}")
    print(f"   Available files in {current_dir}:")
    for f in current_dir.iterdir():
        print(f"      - {f.name}")
else:
    print(f"✅ Config file found")
    try:
        with open(config_path, 'r', encoding='utf-8') as f:
            config_data = yaml.safe_load(f)
        
        print(f"✅ Config loaded successfully")
        
        # Get Global Defaults
        default_models = config_data.get('project_settings', {}).get('default_models', {})
        print(f"📋 Default models: {list(default_models.keys())}")
        
        experiments = config_data.get("experiments", [])
        print(f"📊 Total experiments in config: {len(experiments)}")
        
        enabled_count = sum(1 for e in experiments if e.get("enabled", True))
        print(f"✅ Enabled experiments: {enabled_count}")
        print()
        
        for exp in experiments:
            exp_id = exp.get('experiment_id', 'Unknown')
            is_enabled = exp.get("enabled", True)
            status = "✅ ENABLED" if is_enabled else "⏩ DISABLED"
            print(f"   {status} - {exp_id}")
        
        print()
        print("=" * 70)
        
        run_count = 0
        for exp in experiments:
            if exp.get("enabled", True):
                run_single_experiment(exp, default_models, current_dir)
                run_count += 1
                exp_id_to_disable = exp.get("experiment_id")
                if exp_id_to_disable:
                    with open(config_path, 'r', encoding='utf-8') as f:
                        text = f.read()
                    import re
                    escaped_id = re.escape(exp_id_to_disable)
                    pattern = rf'(experiment_id:\s*"{escaped_id}"\s*\n\s+enabled:)\s*True'
                    text = re.sub(pattern, r'\1 False', text)
                    with open(config_path, 'w', encoding='utf-8') as f:
                        f.write(text)
            else:
                print(f"⏩ Skipping disabled experiment: {exp.get('experiment_id')}")
        
        print("\n" + "=" * 70)
        print(f"🎉 COMPLETE - Ran {run_count} experiment(s)")
        print("=" * 70)
        
    except Exception as e:
        print(f"❌ Error loading/running config: {e}")
        import traceback
        traceback.print_exc()


🚀 STARTING EXPERIMENT RUNNER
📍 Current working directory: /Users/maniamrit/Miami/thesis-work/experiments/Scripts
🔍 Looking for config at: /Users/maniamrit/Miami/thesis-work/experiments/Scripts/master_config.yaml
✅ Config file found
✅ Config loaded successfully
📋 Default models: ['ChatGPT', 'Gemini']
📊 Total experiments in config: 8
✅ Enabled experiments: 1

   ⏩ DISABLED - FewShot_5_V1
   ⏩ DISABLED - FewShot_5_Random_V1
   ⏩ DISABLED - FewShot_5_Random_V2
   ✅ ENABLED - FewShot_10_Random_V1
   ⏩ DISABLED - FewShot_10_Random_V2
   ⏩ DISABLED - FewShot_10_Random_V4
   ⏩ DISABLED - ZeroShot_V1
   ⏩ DISABLED - test_experiment

⏩ Skipping disabled experiment: FewShot_5_V1
⏩ Skipping disabled experiment: FewShot_5_Random_V1
⏩ Skipping disabled experiment: FewShot_5_Random_V2

🔬 --- Starting Experiment: FewShot_10_Random_V1 ---
      📊 Found 10 examples and 90 targets
      📄 Loaded 1 documentation file(s)
      ⏳ Querying ChatGPT...
         > Processing Batch 1/9...
         > Processing B